# Run Selected-10 Failure Cases — NVIDIA NIM API

Repeats the 8-config ablation suite on the 10 obvious failure topics using the **NVIDIA NIM API** (no rate limits, OpenAI-compatible).

**Why NVIDIA instead of Groq?**  
The Groq run hit the 200 k TPD cap mid-way through Stage 2 for the six-agent config. NVIDIA NIM has no daily token limit.

**Model choice for this experiment:**  
The professor asked us to test with a *large* LLM to see if scale fixes the PRO-side skew.

| Model | Params | Notes |
|---|---|---|
| `meta/llama-3.1-405b-instruct` | 405 B | **Recommended** — best reasoning |
| `nvidia/llama-3.1-nemotron-ultra-253b-v1` | 253 B | NVIDIA-tuned, strong instruction following |
| `nvidia/llama-3.1-nemotron-70b-instruct` | 70 B | Faster fallback |

All 10 topics have gold label **CON**; the pipeline previously over-predicted PRO.

Outputs go to `outputs/nvidia_selected10/` (separate from the Groq run).

In [1]:
import json
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "scripts").exists():
        ROOT = candidate
        break

sys.path.append(str((ROOT / "scripts").resolve()))

import groq_selected10_notebook as nb

TOPIC_FILE     = ROOT / "data" / "eval" / "google_form" / "form_topics.jsonl"
OUTPUT_ROOT    = ROOT / "outputs" / "nvidia_selected10"
CACHE_DIR      = OUTPUT_ROOT / "cache"

# ── Model selection ──────────────────────────────────────────────────────────
# Use the env var to override without editing this notebook.
# Default: 405B (best for this experiment; professor asked for a large model).
MODEL = os.environ.get("NVIDIA_MODEL", "meta/llama-3.1-405b-instruct")

# Set FORCE=True to re-run everything even if cached outputs exist.
FORCE          = False
PAIR_BATCH_SIZE = 40

print("Topic file:      ", TOPIC_FILE)
print("Output root:     ", OUTPUT_ROOT)
print("Model:           ", MODEL)
print("NVIDIA_API_KEY set:", bool(os.environ.get("NVIDIA_API_KEY") or os.environ.get("NIM_API_KEY")))

Topic file:       C:\Users\PrajwalBhandary\OneDrive - Cloud Catalyst Asia\Personal\MAJ-Debate\data\eval\google_form\form_topics.jsonl
Output root:      C:\Users\PrajwalBhandary\OneDrive - Cloud Catalyst Asia\Personal\MAJ-Debate\outputs\nvidia_selected10
Model:            meta/llama-3.1-405b-instruct
NVIDIA_API_KEY set: True


In [2]:
# Inspect the 10 topics and estimate how many API requests the suite needs.
topics = nb.load_selected_topics(TOPIC_FILE)
plan   = nb.estimate_request_plan(topics)

print(f"Topics loaded: {len(topics)}")
for t in topics:
    print(f"  {t['topic_id']:15s}  gold={t.get('benchmark_label','?'):3s}  {t['topic_text'][:70]}")

print("\nRequest plan:")
print(json.dumps(plan, indent=2))

Topics loaded: 10
  LOGIC_002        gold=CON  Resolved: If some birds can fly and penguins are birds, then penguins 
  LOGIC_005        gold=CON  Resolved: If it is raining then the ground is wet; the ground is wet, 
  LOGIC_030        gold=CON  Resolved: The number of people who hold a belief is sufficient to dete
  LOGIC_042        gold=CON  Resolved: Humans only use 10 percent of their brains.
  LOGIC_045        gold=CON  Resolved: Antibiotics are effective against viral infections such as t
  DDO_18636        gold=CON  Evolution is not supported by science or evidence
  DDO_50560        gold=CON  Science is a major threat to human existence
  DDO_61182        gold=CON  Teenagers Should Have Unlimited Access to Computers and The Internet
  DDO_70417        gold=CON  This House believes that the United States Federal Government should a
  DDO_10643        gold=CON  children should be allowed to box in school.

Request plan:
{
  "topic_count": 10,
  "minimum_new_request_groups": {
  

In [3]:
# Quick sanity-check: make one test call to confirm the key and model work.
client = nb.NvidiaClient(model=MODEL, cache_dir=CACHE_DIR)
print(f"Using model : {client.model}")
print(f"API keys    : {client.active_key_count} loaded")

resp = client.complete(
    "You are a helpful assistant. Respond only with valid JSON.",
    'Reply with {"status": "ok", "model": "<your model id>"}',
    max_tokens=50,
    namespace="healthcheck",
    use_cache=False,
)
print("Response:", resp["content"])

Using model : meta/llama-3.1-405b-instruct
API keys    : 1 loaded
Response: {"status": "ok", "model": "LLaMA"}


In [ ]:
import time as _time

# Run the full 8-config ablation suite with automatic retry on API failure.
# The suite checkpoints after every topic, so each retry resumes from where it stopped.
MAX_SUITE_RETRIES = 5
RETRY_WAIT_SECONDS = 60

result = None
for _attempt in range(1, MAX_SUITE_RETRIES + 1):
    try:
        result = nb.run_nvidia_selected10_suite(
            model=MODEL,
            topic_file=TOPIC_FILE,
            output_root=OUTPUT_ROOT,
            cache_dir=CACHE_DIR,
            force=FORCE,
            pair_batch_size=PAIR_BATCH_SIZE,
        )
        print(f"Suite completed on attempt {_attempt}.")
        break
    except Exception as _exc:
        print(f"[suite attempt {_attempt}/{MAX_SUITE_RETRIES}] Failed: {_exc}")
        if _attempt < MAX_SUITE_RETRIES:
            print(f"Waiting {RETRY_WAIT_SECONDS}s before retry ...")
            _time.sleep(RETRY_WAIT_SECONDS)
        else:
            print("All retry attempts exhausted. Re-run this cell to try again.")
            raise

if result:
    print(json.dumps(result, indent=2))

[retry 1/5] 400 Client Error: Bad Request for url: https://integrate.api.nvidia.com/v1/chat/completions — waiting 2s …
[retry 2/5] 400 Client Error: Bad Request for url: https://integrate.api.nvidia.com/v1/chat/completions — waiting 4s …
[retry 3/5] 400 Client Error: Bad Request for url: https://integrate.api.nvidia.com/v1/chat/completions — waiting 8s …
[retry 4/5] 400 Client Error: Bad Request for url: https://integrate.api.nvidia.com/v1/chat/completions — waiting 15s …
[suite attempt 1/5] Failed: 400 Client Error: Bad Request for url: https://integrate.api.nvidia.com/v1/chat/completions
Waiting 60s before retry ...
[retry 1/5] 400 Client Error: Bad Request for url: https://integrate.api.nvidia.com/v1/chat/completions — waiting 2s …
[retry 2/5] 400 Client Error: Bad Request for url: https://integrate.api.nvidia.com/v1/chat/completions — waiting 4s …
[retry 3/5] 400 Client Error: Bad Request for url: https://integrate.api.nvidia.com/v1/chat/completions — waiting 8s …
[retry 4/5] 400 C

In [ ]:
# Display the ablation summary table.
summary_rows = result["summary_rows"]
try:
    import pandas as pd
    df = pd.DataFrame(summary_rows)
    display(df)
except Exception:
    print(json.dumps(summary_rows, indent=2))

In [ ]:
# Compare NVIDIA results vs the previous Groq run (if it exists).
groq_root = ROOT / "outputs" / "groq_selected10"
nvidia_root = ROOT / "outputs" / "nvidia_selected10"

configs = ["single_llm", "cot", "direct_judge", "two_agents",
           "six_agents", "targeted_attacks", "dung_no_agents", "full"]

comparison = []
for config in configs:
    row = {"config": config}
    for label, root_dir in [("groq", groq_root), ("nvidia", nvidia_root)]:
        path = root_dir / "stage4" / config / "stage4_judgments.json"
        if path.exists():
            doc = json.loads(path.read_text(encoding="utf-8"))
            js  = doc.get("judgments", [])
            row[f"{label}_accuracy"] = nb.agreement_pct(js)
            row[f"{label}_n"]        = len(js)
        else:
            row[f"{label}_accuracy"] = None
            row[f"{label}_n"]        = 0
    comparison.append(row)

try:
    import pandas as pd
    display(pd.DataFrame(comparison))
except Exception:
    print(json.dumps(comparison, indent=2))

In [ ]:
# Print the human-readable summary report.
summary_path = Path(result["summary_json"])
md_path = summary_path.with_suffix(".md")
if md_path.exists():
    print(md_path.read_text(encoding="utf-8"))
else:
    print(summary_path.read_text(encoding="utf-8"))